# Smart Classroom Energy System — Secondary Dataset Decision Tree

Reproducible analysis notebook using the project-supplied `smart_lighting_dataset_2024.csv`. Numeric target codes remain as provided (0/1/2); meanings not assumed.

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
DATA_PATH=Path('data/smart_lighting_dataset_2024.csv')
df=pd.read_csv(DATA_PATH)
df.shape, df.columns.tolist(), df.isna().sum().sum(), df.duplicated().sum()

((12000, 18), ['timestamp', 'zone_id', 'ambient_light_lux', 'motion_detected', 'temperature_celsius', 'occupancy_count', 'day_of_week', 'time_of_day', 'weather_condition', 'special_event_flag', 'energy_price_per_kwh', 'prev_hour_energy_usage_kwh', 'traffic_density', 'avg_pedestrian_speed', 'adjusted_light_intensity', 'energy_consumption_kwh', 'lighting_action_class', 'hour_of_day'], 0, 0)

## Feature engineering and target
Extract hour from timestamp. Use the 11 predictors below; target remains `lighting_action_class`.

In [2]:
df['hour_of_day']=pd.to_datetime(df['timestamp'],errors='coerce').dt.hour
FEATURES=['ambient_light_lux', 'motion_detected', 'temperature_celsius', 'occupancy_count', 'hour_of_day', 'day_of_week', 'time_of_day', 'weather_condition', 'special_event_flag', 'traffic_density', 'avg_pedestrian_speed']
TARGET='lighting_action_class'
df=df.dropna(subset=FEATURES+[TARGET]).copy()
X=df[FEATURES]; y=df[TARGET].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
NUMERIC_FEATURES=['ambient_light_lux', 'motion_detected', 'temperature_celsius', 'occupancy_count', 'hour_of_day', 'special_event_flag', 'traffic_density', 'avg_pedestrian_speed']
CATEGORICAL_FEATURES=['day_of_week', 'time_of_day', 'weather_condition']
preprocessor=ColumnTransformer([('categorical',OneHotEncoder(handle_unknown='ignore'),CATEGORICAL_FEATURES),('numeric','passthrough',NUMERIC_FEATURES)])
model=Pipeline([('preprocessor',preprocessor),('classifier',DecisionTreeClassifier(criterion='gini',max_depth=5,min_samples_leaf=10,class_weight='balanced',random_state=42))])
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
accuracy_score(y_test,y_pred),classification_report(y_test,y_pred,labels=[0,1,2],zero_division=0)

0.8329
              precision    recall  f1-score   support

           0       0.74      0.77      0.76       798
           1       0.89      0.94      0.91      1419
           2       0.75      0.28      0.41       183

    accuracy                           0.83      2400
   macro avg       0.79      0.66      0.69      2400
weighted avg       0.83      0.83      0.82      2400


## Confusion matrix
Rows = actual classes; columns = predicted classes; label order 0, 1, 2.

In [3]:
cm=confusion_matrix(y_test,y_pred,labels=[0,1,2])
ConfusionMatrixDisplay(cm,display_labels=['Class 0','Class 1','Class 2']).plot(cmap='Blues',values_format='d')
plt.title('Confusion Matrix — Held-out Test Set'); plt.tight_layout(); plt.show()
cm

[[ 615  171   12]
 [  81 1333    5]
 [ 131    1   51]]

## Label transparency and limitations
No new labels are created for this supervised model: `lighting_action_class` is used as provided. The separate project ON/OFF rule is LIGHT ON iff motion=1, hour 7–18 inclusive, lux<300; otherwise LIGHT OFF. Verify the numeric codes’ semantics and the supplied CSV lineage against the Kaggle source page before claiming direct provenance or mapping classes to lamp actions. The test split metrics are not proof of real classroom or relay performance.

In [4]:
joblib.dump({'model':model,'features':FEATURES,'target':TARGET,'class_codes':[0,1,2],'dataset_note':'Supplied CSV; class code semantics unverified','evaluation_random_state':42},Path('models/smart_classroom_decision_tree_kaggle.joblib'))
print('Model artifact saved.')

Model artifact saved.
